# HAM10000 ResNet Fine-tuning
This notebook demonstrates how to fine-tune a ResNet model on the HAM10000 dataset using the `ResNetFineTuneTrainer` class.

In [1]:
import sys
import os

# Add the project root to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from dualbranch.ham10000_dataset_manager import HAM10000DatasetManager
from dualbranch.ResNetFinetuneTrain import ResNetFineTuneTrainer

In [2]:
# 1. Prepare the Data
# Initialize the dataset manager
data_dir = r"C:\Users\z004ps0b\Documents\dermascan\data"
manager = HAM10000DatasetManager(data_dir=data_dir)

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
manager.load_cleaned_metadata(filter_melanoma_only=True)

# Split the data into train (70%), validation (15%), and test (15%) sets
train_df, val_df, test_df = manager.train_test_val_split(val_size=0.15, test_size=0.15)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Training samples: 2766
Validation samples: 593
Test samples: 593


In [3]:
# 2. Configure the Trainer
trainer = ResNetFineTuneTrainer()

# Set up the model (ResNet50, 2 classes: nv vs mel)
trainer.set_model(model_name='resnet50', pretrained=True)

# Set data loaders
trainer.set_data_loaders(train_df, val_df, test_df)

# Configure optimizer and scheduler
trainer.configure_optimizer(optimizer_type='adam', lr=1e-4, weight_decay=1e-4)
trainer.configure_scheduler(scheduler_type='reduce_on_plateau', patience=3)

# Set output paths for artifacts
trainer.set_output_paths(
    model_save_dir="../results/models",
    results_csv_path="../results/training_results.csv",
    history_plot_dir="../results/plots"
)

c:\Users\z004ps0b\Documents\dermascan\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\z004ps0b\Documents\dermascan\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
# 3. Train the Model
# Train for a specified number of epochs
trainer.train(epochs=15)

Epoch 1/15
----------
train Loss: 0.4827 Acc: 0.8019
train Loss: 0.4827 Acc: 0.8019
val Loss: 0.4276 Acc: 0.8280
Epoch 2/15
----------
val Loss: 0.4276 Acc: 0.8280
Epoch 2/15
----------
train Loss: 0.4079 Acc: 0.8308
train Loss: 0.4079 Acc: 0.8308
val Loss: 0.3895 Acc: 0.8331
Epoch 3/15
----------
val Loss: 0.3895 Acc: 0.8331
Epoch 3/15
----------
train Loss: 0.3806 Acc: 0.8348
train Loss: 0.3806 Acc: 0.8348
val Loss: 0.3648 Acc: 0.8331
Epoch 4/15
----------
val Loss: 0.3648 Acc: 0.8331
Epoch 4/15
----------
train Loss: 0.3619 Acc: 0.8359
train Loss: 0.3619 Acc: 0.8359
val Loss: 0.3476 Acc: 0.8398
Epoch 5/15
----------
val Loss: 0.3476 Acc: 0.8398
Epoch 5/15
----------
train Loss: 0.3468 Acc: 0.8471
train Loss: 0.3468 Acc: 0.8471
val Loss: 0.3367 Acc: 0.8364
Epoch 6/15
----------
val Loss: 0.3367 Acc: 0.8364
Epoch 6/15
----------
train Loss: 0.3417 Acc: 0.8536
train Loss: 0.3417 Acc: 0.8536
val Loss: 0.3254 Acc: 0.8600
Epoch 7/15
----------
val Loss: 0.3254 Acc: 0.8600
Epoch 7/15
-----

In [5]:
# 4. Evaluate and Save Results
trainer.evaluate()
trainer.save_results_to_csv()

Evaluation results - F1: 0.8396, AUC: 0.8508, Accuracy: 0.8499, Sensitivity: 0.4216, Specificity: 0.9389
